# Exercise 1: Prompt Chaining for a Customer Support AI

**Tool:** Google Gemini API (`gemini-3.8-flash`) in Google Colab

**Chain:** (1) classify the message, (2) extract details and find missing information, (3) apply an escalation rule, (4) draft a reply, (5) review the draft against policy and polish it. Each step's output feeds the next step.

In [ ]:
%pip install -q google-genai

import os
import re
import json
import textwrap
import traceback
import random
from google import genai
from google.genai import types

# Load the API key (Colab secret named GEMINI_API_KEY, or an environment variable locally)
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

client = genai.Client()
MODEL = "gemini-3.8-flash"

# Helper: send one prompt to the model and return the reply text
def ask(prompt, system=None):
    config = types.GenerateContentConfig(system_instruction=system)
    return client.models.generate_content(model=MODEL, contents=prompt, config=config).text


In [ ]:
# Step 1 prompt: classify the message
CLASSIFY_PROMPT = """You are a support triage assistant for an online store called ShopEase.
Read the customer message and return ONLY valid JSON with these keys:
  "category": one of "billing", "shipping", "technical", "returns", "other"
  "sentiment": one of "angry", "frustrated", "neutral", "positive"
  "urgency": one of "low", "medium", "high"

Customer message:
{message}"""

# Step 2 prompt: extract key details
EXTRACT_PROMPT = """You are extracting facts from a customer support message (category: {category}).
Return ONLY valid JSON with these keys. Use null for anything the customer did not state. Do not guess.
  "order_number": string or null
  "product": string or null
  "issue_summary": one short sentence
  "requested_action": string or null
  "missing_info": list of strings naming details still needed from the customer to resolve this (empty list if none)

Customer message:
{message}"""

# Step 4 prompt: draft the reply
DRAFT_PROMPT = """You are a friendly, professional ShopEase support agent.
Write a reply to the customer using the context below.

Category: {category}
Customer sentiment: {sentiment}
Urgency: {urgency}
Extracted details: {details}
Escalation: {escalation}

Rules:
- Open by acknowledging the customer's feeling in one sentence (more empathy if angry or frustrated).
- Refer to the specific order and product when they are known.
- If missing_info is not empty, politely ask for those details.
- If Escalation is not "None", follow it.
- State one clear next step.
- Do not promise refunds or delivery dates; say the team will confirm after reviewing the order.
- Maximum 120 words.

Original message:
{message}"""

# Step 5 prompt: review the draft against policy and polish it
REVIEW_PROMPT = """You are a support quality reviewer. Check the draft reply against these policies:
1. No promised refunds, credits or specific delivery dates.
2. Polite, no blame on the customer, no jargon.
3. Mentions the order number and product if they are known: {details}
4. Contains exactly one clear next step.
5. 120 words or fewer.

Fix any violations and improve the tone. Return ONLY the final reply text, with no commentary.

Draft reply:
{draft}"""

In [ ]:
# Sample customer messages
sample_messages = [
    "I ordered the Aurora wireless headphones (order #48213) two weeks ago and tracking still says 'label created'. "
    "This is the third time I've written and nobody has helped me. I need these for a trip on Friday!",

    "Hi, I was charged twice for my subscription this month. The two charges of $14.99 both show on my card. "
    "Can you fix this please?",

    "The app keeps crashing every time I open my order history. I'm on iPhone and updated yesterday. Love the products though!",
]

In [ ]:
# Run the prompt chain on each message
final_replies = []
for message in sample_messages:
    print("CUSTOMER MESSAGE:")
    print(textwrap.fill(message, 90))

    # Step 1: classify the message (JSON output)
    reply = ask(CLASSIFY_PROMPT.format(message=message))
    triage = json.loads(re.sub(r"^```(?:json)?|```$", "", reply.strip(), flags=re.MULTILINE).strip())
    print("\n[Step 1] Triage:", triage)

    # Step 2: extract details, using the category from step 1
    reply = ask(EXTRACT_PROMPT.format(message=message, category=triage["category"]))
    details = json.loads(re.sub(r"^```(?:json)?|```$", "", reply.strip(), flags=re.MULTILINE).strip())
    print("[Step 2] Details:", details)

    # Step 3: escalation rule, using the urgency and sentiment from step 1
    if triage["urgency"] == "high" and triage["sentiment"] in ("angry", "frustrated"):
        escalation = "Escalate: tell the customer their case is being passed to a senior support specialist."
    else:
        escalation = "None"
    print("[Step 3] Escalation:", escalation)

    # Step 4: draft a reply, using the outputs of steps 1, 2 and 3
    draft = ask(DRAFT_PROMPT.format(message=message, details=json.dumps(details), escalation=escalation, **triage))
    print("\n[Step 4] Draft reply:")
    print(draft)

    # Step 5: review the draft against policy and polish it
    final_reply = ask(REVIEW_PROMPT.format(details=json.dumps(details), draft=draft))
    final_replies.append(final_reply)
    print("\n[Step 5] FINAL reply:")
    print(final_reply)
    print("\n" + "=" * 90 + "\n")

In [ ]:
# Iteration: compare a first-attempt prompt (v1) with the refined chain (v2) on the same message
test_message = sample_messages[0]

# v1: a single prompt with no role, constraints or context
v1_reply = ask("Write a reply to this customer message:\n" + test_message)
print("BEFORE (v1: single prompt, no constraints):")
print(v1_reply)

# v2: the final reply produced by the five-step chain above
print("\nAFTER (v2: five-step chain):")
print(final_replies[0])

# Compare length
print("\nWords: v1 =", len(v1_reply.split()), "| v2 =", len(final_replies[0].split()))

**Iteration notes:** v1 was a single prompt with no role, tone rules or policy limits. v2 splits the work into five steps, adds the policy constraints (no promised refunds or dates, one next step, 120 words maximum), asks for missing details, and applies an escalation rule. Compare the BEFORE and AFTER outputs above.

**Reflection:** each prompt has one job, so every output is easy to check. Step 4 adapts its tone to the sentiment from step 1 and uses the facts from step 2 and the escalation from step 3, and step 5 catches policy violations (such as promising a refund) before the customer sees the reply.